## Шаг 1. Загрузка данных и первичный анализ

Загружаем итоговый датасет `merged_buildings_by_geometry.parquet`, полученный после сопоставления источников А и Б.

В датасете:
- `target_height` — целевая переменная (высота здания)
- `target_height_is_observed` — флаг наличия высоты (1 — есть, 0 — нет)
- `target_height_reliability` — надёжность наблюдения (high / medium / low / none)
- числовые признаки: этажность, площадь, характеристики соседей
- категориальный признак: `mode_purpose_b` (тип здания)

**Задача:** подготовить данные для обучения линейной регрессии с пространственной кросс-валидацией.

In [3]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
import seaborn as sns
from pathlib import Path 

DATA_DIR = Path("../data/interim/")

# Загружаем датасет
df = pd.read_parquet(DATA_DIR / 'merged_buildings_by_geometry.parquet')

print(f"Всего записей: {len(df):,}")
print(f"Колонки: {df.columns.tolist()}")
print(df.head())

Всего записей: 139,649
Колонки: ['component_id', 'match_type', 'match_confidence', 'geometry_source', 'target_height', 'target_height_is_observed', 'target_height_source', 'target_height_source_detail', 'target_height_reliability', 'n_a', 'n_b', 'uids_a', 'uids_b', 'n_edges_ab', 'max_iou', 'mean_iou', 'max_overlap_a', 'max_overlap_b', 'min_dist_m', 'sum_area_a', 'sum_area_b', 'union_area_a', 'union_area_b', 'union_area_all', 'n_b_with_height', 'median_height_b', 'median_stairs_b', 'median_avg_floor_height_b', 'mode_purpose_b', 'n_neighbors_50m', 'n_neighbors_obs_height_50m', 'neighbor_height_mean_50m', 'neighbor_height_median_50m', 'neighbor_height_min_50m', 'neighbor_height_max_50m', 'neighbor_height_std_50m', 'neighbor_height_q25_50m', 'neighbor_height_q75_50m', 'n_neighbors_100m', 'n_neighbors_obs_height_100m', 'neighbor_height_mean_100m', 'neighbor_height_median_100m', 'neighbor_height_min_100m', 'neighbor_height_max_100m', 'neighbor_height_std_100m', 'neighbor_height_q25_100m', 'n

## Шаг 2. Фильтрация данных и подготовка к обучению

Для обучения модели оставляем только здания с известной высотой (`target_height_is_observed == 1`). Включаем все уровни надёжности (`high`, `medium`, `low`), чтобы не терять данные.

`rep_geometry` при загрузке через `pd.read_parquet()` сохраняется в бинарном формате WKB (Well-Known Binary). Для работы с геометрией преобразуем его в `shapely.geometry` с помощью `shapely.wkb.loads()`.
Затем извлекаем центроиды геометрий для пространственной кросс-валидации. Центроид — это геометрический центр здания, который позволит нам разбить данные по пространственным группам (сетке 1000×1000 м). 


**Зачем это нужно:**  
Если не учитывать пространственную структуру, в обучающую и тестовую выборки могут попасть здания из одного квартала. Поскольку высота зданий коррелирует с районом, модель может «подглядеть» закономерности и показать завышенное качество. такое явление называется пространственным переобучением. Spatial CV предотвращает это.


In [4]:
from shapely import wkb

# Оставляем только здания с известной высотой
df_train = df[df['target_height_is_observed'] == 1].copy()
print(f"Зданий с известной высотой: {len(df_train):,}")

# Преобразуем бинарную геометрию WKB в shapely.geometry
def wkb_to_geometry(x):
    try:
        return wkb.loads(x) if isinstance(x, bytes) else x
    except:
        return None

df_train['geometry'] = df_train['rep_geometry'].apply(wkb_to_geometry)
print(f"Геометрия преобразована. Проверка: {df_train['geometry'].notna().sum()} из {len(df_train)}")

# Создаём GeoDataFrame
gdf_train = gpd.GeoDataFrame(df_train, geometry='geometry', crs='EPSG:32635')
print(f"GeoDataFrame создан, CRS: {gdf_train.crs}")

# Извлекаем центроиды
gdf_train['centroid'] = gdf_train.geometry.centroid
gdf_train['centroid_x'] = gdf_train['centroid'].x
gdf_train['centroid_y'] = gdf_train['centroid'].y

print(f"Центроиды добавлены. Пример координат:")
print(gdf_train[['centroid_x', 'centroid_y']].head())

# Проверяем, что центроиды есть у всех
print(f"Пропусков в центроидах: {gdf_train['centroid'].isna().sum()}")

Зданий с известной высотой: 101,563
Геометрия преобразована. Проверка: 101563 из 101563
GeoDataFrame создан, CRS: EPSG:32635
Центроиды добавлены. Пример координат:
      centroid_x    centroid_y
0  673859.539664  6.635505e+06
1  673885.218595  6.635511e+06
2  677022.530648  6.640407e+06
5  678532.626405  6.638733e+06
6  678893.429480  6.640099e+06
Пропусков в центроидах: 0


## Шаг 3. Создание пространственных групп для spatial CV

Для предотвращения пространственного переобучения разбиваем данные на группы по сетке координат. Каждая группа — это квадрат 1000×1000 м. Здания из одной группы не будут попадать одновременно в обучающую и тестовую выборки.

**Почему 1000 м:**  
Это размер типичного квартала в городе. Соседние здания часто имеют схожую высоту, поэтому важно, чтобы они не оказались в разных folds.

**Что делаем:**
1. Округляем координаты центроида до сетки 1000 м
2. Объединяем округлённые x и y в уникальный идентификатор группы
3. Эти группы будут использоваться в GroupKFold

In [5]:
# Размер ячейки сетки (метры)
GRID_SIZE = 1000

# Создаём группы по сетке
gdf_train['grid_x'] = (gdf_train['centroid_x'] / GRID_SIZE).round().astype(int)
gdf_train['grid_y'] = (gdf_train['centroid_y'] / GRID_SIZE).round().astype(int)
gdf_train['spatial_group'] = gdf_train['grid_x'].astype(str) + '_' + gdf_train['grid_y'].astype(str)

# Статистика по группам
n_groups = gdf_train['spatial_group'].nunique()
avg_per_group = len(gdf_train) / n_groups

print(f"Размер сетки: {GRID_SIZE} м")
print(f"Всего пространственных групп: {n_groups:,}")
print(f"Среднее количество зданий в группе: {avg_per_group:.1f}")
print(f"Минимум зданий в группе: {gdf_train['spatial_group'].value_counts().min()}")
print(f"Максимум зданий в группе: {gdf_train['spatial_group'].value_counts().max()}")

# Показываем пример групп
print("\nПример групп (первые 10):")
print(gdf_train[['centroid_x', 'centroid_y', 'spatial_group']].head(10))

Размер сетки: 1000 м
Всего пространственных групп: 664
Среднее количество зданий в группе: 153.0
Минимум зданий в группе: 1
Максимум зданий в группе: 1625

Пример групп (первые 10):
       centroid_x    centroid_y spatial_group
0   673859.539664  6.635505e+06      674_6636
1   673885.218595  6.635511e+06      674_6636
2   677022.530648  6.640407e+06      677_6640
5   678532.626405  6.638733e+06      679_6639
6   678893.429480  6.640099e+06      679_6640
11  679039.385738  6.659670e+06      679_6660
12  679519.733590  6.650681e+06      680_6651
14  679067.324380  6.660142e+06      679_6660
16  679538.545110  6.654101e+06      680_6654
17  679528.316752  6.653944e+06      680_6654


## Шаг 4. Подготовка признаков (OneHot + StandardScaler)

Чтобы не повторять препроцессинг для каждого бейзлайна, подготовим данные один раз:
1. **One-Hot кодирование** для категориального признака `mode_purpose_b`
2. **StandardScaler** для всех числовых признаков

**Почему StandardScaler:** линейная регрессия чувствительна к масштабу признаков. Приводим все числовые признаки к среднему 0 и стандартному отклонению 1.

**Важно:** масштабирование выполняется после фильтрации данных (только на обучающей выборке), чтобы не было утечки данных из тестовой выборки.

**Примеры числовых признаков:**
- `median_stairs_b` — этажность
- `median_avg_floor_height_b` — средняя высота этажа
- `union_area_all` — площадь здания
- `n_a`, `n_b` — количество полигонов
- `neighbor_height_mean_50m`, `neighbor_height_mean_100m` — средняя высота соседей

In [6]:
# Определяем колонки
cat_col = 'mode_purpose_b'

# Список всех числовых признаков (кроме целевой и технических)
num_cols_all = [
    'median_stairs_b', 'median_avg_floor_height_b', 'union_area_all',
    'n_a', 'n_b', 'n_edges_ab',
    'max_iou', 'mean_iou', 'max_overlap_a', 'max_overlap_b', 'min_dist_m',
    'n_b_with_height',
    'n_neighbors_50m', 'n_neighbors_obs_height_50m',
    'neighbor_height_mean_50m', 'neighbor_height_median_50m',
    'neighbor_height_min_50m', 'neighbor_height_max_50m',
    'neighbor_height_std_50m', 'neighbor_height_q25_50m', 'neighbor_height_q75_50m',
    'n_neighbors_100m', 'n_neighbors_obs_height_100m',
    'neighbor_height_mean_100m', 'neighbor_height_median_100m',
    'neighbor_height_min_100m', 'neighbor_height_max_100m',
    'neighbor_height_std_100m', 'neighbor_height_q25_100m', 'neighbor_height_q75_100m'
]

# Проверяем, какие колонки есть в данных
available_num_cols = [col for col in num_cols_all if col in gdf_train.columns]
print(f"Доступно числовых признаков: {len(available_num_cols)}")

# OneHot для категориального признака
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_encoded = encoder.fit_transform(gdf_train[[cat_col]])
cat_cols = encoder.get_feature_names_out([cat_col])
cat_df = pd.DataFrame(cat_encoded, columns=cat_cols, index=gdf_train.index)

print(f"OneHot создано колонок: {len(cat_cols)}")

# StandardScaler для числовых признаков
scaler = StandardScaler()
num_scaled = scaler.fit_transform(gdf_train[available_num_cols])
num_df = pd.DataFrame(num_scaled, columns=available_num_cols, index=gdf_train.index)


# Объединяем всё в один DataFrame
X_prepared = pd.concat([num_df, cat_df], axis=1)

print(f"\nИтоговое количество признаков: {X_prepared.shape[1]}")
print(f"Пример первых 5 строк:")
print(X_prepared.head())

Доступно числовых признаков: 30
OneHot создано колонок: 17

Итоговое количество признаков: 47
Пример первых 5 строк:
   median_stairs_b  median_avg_floor_height_b  union_area_all       n_a  \
0        -0.516866                   0.824816        0.240468  2.937679   
1        -0.516866                   0.824816        0.202824  1.623461   
2         5.062060                  -1.230328        3.545377  4.689970   
5         3.887549                  -1.230328        1.344212  7.318407   
6         5.355687                  -1.230328       -0.044001  0.747316   

        n_b  n_edges_ab   max_iou  mean_iou  max_overlap_a  max_overlap_b  \
0 -0.282261    1.201544 -1.802793 -1.510424       1.310364      -2.389233   
1 -0.282261    0.595128 -1.572913 -1.357019       1.405731      -2.181156   
2  3.096949    5.042182 -0.051075 -1.341646       1.457137       1.243860   
5  3.579693    5.850737  1.323307 -0.919775       1.457137       1.243860   
6  0.200483    0.595128  0.684973 -0.740086    

## Шаг 5. Тестирование бейзлайнов с разными наборами признаков

Обучаем линейную регрессию на 6 наборах признаков, добавляя их последовательно. Для каждого набора запускаем 5-fold spatial CV (GroupKFold) и считаем метрики:

- **RMSE** (Root Mean Square Error) — среднеквадратичная ошибка в метрах. Штрафует большие ошибки.
- **MAE** (Mean Absolute Error) — средняя абсолютная ошибка в метрах.
- **MSE** (Mean Squared Error) — среднеквадратичная ошибка (в м²). Важна для расчёта RMSE.
- **R²** — доля объяснённой дисперсии (чем ближе к 1, тем лучше).


**Порядок тестирования:**
1. **Baseline** — этажность, высота этажа, площадь, тип здания
2. **+ n_a/n_b** — добавляем количество полигонов
3. **+ neighbors_mean** — добавляем среднюю высоту соседей (50м и 100м)
4. **+ neighbors_full_50m** — добавляем все признаки соседей в радиусе 50м
5. **+ neighbors_full_100m** — добавляем все признаки соседей в радиусе 100м
6. **Full** — все доступные числовые признаки

#### Шаг 5.1. Определение наборов признаков

Разбиваем все признаки на логические группы для последовательного тестирования.

In [7]:
# Базовые колонки (этажность, высота этажа, площадь)
base_cols = ['median_stairs_b', 'median_avg_floor_height_b', 'union_area_all']

# Колонки one-hot для типа здания
onehot_cols = [c for c in X_prepared.columns if c.startswith('mode_purpose_b_')]

# Количество полигонов
n_a_b_cols = ['n_a', 'n_b']

# Средняя высота соседей
neighbors_mean_cols = ['neighbor_height_mean_50m', 'neighbor_height_mean_100m']

# Все признаки соседей 50м
neighbors_50m_cols = [
    'n_neighbors_50m', 'n_neighbors_obs_height_50m',
    'neighbor_height_mean_50m', 'neighbor_height_median_50m',
    'neighbor_height_min_50m', 'neighbor_height_max_50m',
    'neighbor_height_std_50m', 'neighbor_height_q25_50m', 'neighbor_height_q75_50m'
]

# Все признаки соседей 100м
neighbors_100m_cols = [
    'n_neighbors_100m', 'n_neighbors_obs_height_100m',
    'neighbor_height_mean_100m', 'neighbor_height_median_100m',
    'neighbor_height_min_100m', 'neighbor_height_max_100m',
    'neighbor_height_std_100m', 'neighbor_height_q25_100m', 'neighbor_height_q75_100m'
]

# Остальные признаки (метрики сопоставления и прочее)
other_cols = [c for c in X_prepared.columns 
              if c not in base_cols + onehot_cols + n_a_b_cols + neighbors_mean_cols + neighbors_50m_cols + neighbors_100m_cols]

print(f"Базовые: {len(base_cols)}")
print(f"One-hot (тип здания): {len(onehot_cols)}")
print(f"n_a/n_b (количество полигонов): {len(n_a_b_cols)}")
print(f"neighbors_mean (средняя высота соседей): {len(neighbors_mean_cols)}")
print(f"neighbors_50m (все признаки соседей 50м): {len(neighbors_50m_cols)}")
print(f"neighbors_100m (все признаки соседей 100м): {len(neighbors_100m_cols)}")
print(f"Другие признаки: {len(other_cols)}")

# Формируем наборы признаков
feature_sets = {
    '1_Baseline': base_cols + onehot_cols,
    '2_+ n_a_n_b': base_cols + onehot_cols + n_a_b_cols,
    '3_+ neighbors_mean': base_cols + onehot_cols + neighbors_mean_cols,
    '4_+ neighbors_50m': base_cols + onehot_cols + neighbors_50m_cols,
    '5_+ neighbors_100m': base_cols + onehot_cols + neighbors_50m_cols + neighbors_100m_cols,
    '6_Full': base_cols + onehot_cols + n_a_b_cols + neighbors_50m_cols + neighbors_100m_cols + other_cols
}

print("\nНаборы признаков сформированы:")
for name, cols in feature_sets.items():
    print(f"  {name}: {len(cols)} признаков")

Базовые: 3
One-hot (тип здания): 17
n_a/n_b (количество полигонов): 2
neighbors_mean (средняя высота соседей): 2
neighbors_50m (все признаки соседей 50м): 9
neighbors_100m (все признаки соседей 100м): 9
Другие признаки: 7

Наборы признаков сформированы:
  1_Baseline: 20 признаков
  2_+ n_a_n_b: 22 признаков
  3_+ neighbors_mean: 22 признаков
  4_+ neighbors_50m: 29 признаков
  5_+ neighbors_100m: 38 признаков
  6_Full: 47 признаков


#### Шаг 5.2. Функция для spatial кросс-валидации

Создаём функцию `evaluate_model_cv`, которая:
- принимает признаки, целевую переменную, пространственные группы и модель
- разбивает данные на 5 folds с помощью GroupKFold (здания из одной группы не попадают в разные folds)
- для каждого fold обучает модель и считает метрики (RMSE, MAE, MSE, R²)
- возвращает средние значения и стандартные отклонения

In [8]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

def evaluate_model_cv(X, y, groups, model, cv_folds=5):
    """
    Оценивает модель с spatial CV (GroupKFold)
    
    Параметры:
    - X: признаки
    - y: целевая переменная
    - groups: пространственные группы
    - model: модель для обучения
    - cv_folds: количество folds
    
    Возвращает словарь с метриками
    """
    group_kfold = GroupKFold(n_splits=cv_folds)
    
    rmse_list = []
    mae_list = []
    mse_list = []
    r2_list = []
    
    for fold, (train_idx, test_idx) in enumerate(group_kfold.split(X, y, groups), 1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        
        rmse_list.append(rmse)
        mae_list.append(mae)
        mse_list.append(mse)
        r2_list.append(r2)
        
        print(f"  Fold {fold}: RMSE={rmse:.3f} м, MAE={mae:.3f} м, R²={r2:.3f}")
    
    return {
        'RMSE_mean': np.mean(rmse_list),
        'RMSE_std': np.std(rmse_list),
        'MAE_mean': np.mean(mae_list),
        'MSE_mean': np.mean(mse_list),
        'R2_mean': np.mean(r2_list)
    }



#### Шаг 5.3. Проверка пропусков в данных

Перед обучением моделей необходимо проверить наличие пропусков в подготовленных данных. Linear Regression не поддерживает значения NaN, поэтому важно выявить все пропуски и определить стратегию их обработки.

**Результат проверки:**
- Всего признаков: 47
- Признаков с пропусками: 19
- Целевая переменная (`target_height`) пропусков не имеет

Обнаруженные пропуски относятся к двум группам:

1. **Метрики сопоставления** (`max_iou`, `mean_iou`, `max_overlap_a`, `max_overlap_b`, `min_dist_m`) — 27 984 пропуска (27.55%).
   Эти признаки отсутствуют для компонент типа `A_only` и `B_only`, где нет пары между источниками.

2. **Статистики высоты соседей** (`neighbor_height_*`) — от 189 до 3 838 пропусков (0.19%–3.78%).
   NaN здесь означает отсутствие соседского контекста: нет соседей в радиусе или у соседей нет наблюдаемой высоты.

Для корректного обучения необходимо обработать эти пропуски, сохраняя при этом всю доступную информацию.

In [9]:
# Определяем целевую переменную и группы
y = gdf_train['target_height']
groups = gdf_train['spatial_group']

# Проверяем пропуски в X_prepared
nan_counts = X_prepared.isna().sum()
nan_cols = nan_counts[nan_counts > 0]

print(f"Всего признаков: {X_prepared.shape[1]}")
print(f"Признаков с пропусками: {len(nan_cols)}")

if len(nan_cols) > 0:
    print("\nПризнаки с пропусками:")
    for col, count in nan_cols.items():
        percent = count / len(X_prepared) * 100
        print(f"  {col}: {count} пропусков ({percent:.2f}%)")
else:
    print("Пропусков в признаках нет!")

# Проверяем целевую переменную
y_nan = y.isna().sum()
if y_nan > 0:
    print(f"\nЦелевая переменная: {y_nan} пропусков")
else:
    print(f"\nЦелевая переменная: пропусков нет")



Всего признаков: 47
Признаков с пропусками: 19

Признаки с пропусками:
  max_iou: 27984 пропусков (27.55%)
  mean_iou: 27984 пропусков (27.55%)
  max_overlap_a: 27984 пропусков (27.55%)
  max_overlap_b: 27984 пропусков (27.55%)
  min_dist_m: 27984 пропусков (27.55%)
  neighbor_height_mean_50m: 1017 пропусков (1.00%)
  neighbor_height_median_50m: 1017 пропусков (1.00%)
  neighbor_height_min_50m: 1017 пропусков (1.00%)
  neighbor_height_max_50m: 1017 пропусков (1.00%)
  neighbor_height_std_50m: 3838 пропусков (3.78%)
  neighbor_height_q25_50m: 1017 пропусков (1.00%)
  neighbor_height_q75_50m: 1017 пропусков (1.00%)
  neighbor_height_mean_100m: 189 пропусков (0.19%)
  neighbor_height_median_100m: 189 пропусков (0.19%)
  neighbor_height_min_100m: 189 пропусков (0.19%)
  neighbor_height_max_100m: 189 пропусков (0.19%)
  neighbor_height_std_100m: 553 пропусков (0.54%)
  neighbor_height_q25_100m: 189 пропусков (0.19%)
  neighbor_height_q75_100m: 189 пропусков (0.19%)

Целевая переменная: проп

#### Шаг 5.4. Обработка пропусков

Для обработки пропусков применяется стратегия, учитывающая природу каждого типа признаков.

1. Метрики сопоставления (`max_iou`, `mean_iou`, `max_overlap_a`, `max_overlap_b`, `min_dist_m`)

Эти признаки описывают качество сопоставления источников А и Б, а не свойства самого здания. Для предсказания высоты они неинформативны и могут создавать ложные корреляции.  
**Решение:** исключаем эти признаки из набора полностью.

2. Статистики высоты соседей (`neighbor_height_mean_*`, `neighbor_height_median_*`, `neighbor_height_min_*`, `neighbor_height_max_*`, `neighbor_height_std_*`, `neighbor_height_q25_*`, `neighbor_height_q75_*`)

NaN здесь означает отсутствие соседского контекста:
- нет соседей в заданном радиусе;
- или соседи есть, но у них нет наблюдаемой высоты.

Это само по себе информация, которую нельзя терять.

**Как обрабатываем:**
- `n_neighbors_*` и `n_neighbors_obs_height_*` — оставляем как есть. Значение 0 означает отсутствие соседей.
- Для каждой статистической характеристики соседей создаём дополнительный индикатор `has_neighbor_stats_*`, который показывает, доступно ли значение (1 — есть соседи с высотой, 0 — нет). По индикатору модель поймет, что это не реальная высота, а отсутствие данных.
- Сами признаки заполняем техническим значением 0.

Такой подход позволяет модели различать два сценария: отсутствие соседей и наличие соседей без данных о высоте, сохраняя всю информацию о локальном контексте.

In [10]:
# Удаляем метрики сопоставления (неинформативны для предсказания высоты)
cols_to_drop = ['max_iou', 'mean_iou', 'max_overlap_a', 'max_overlap_b', 'min_dist_m']
existing_cols = [c for c in cols_to_drop if c in X_prepared.columns]
X_prepared = X_prepared.drop(columns=existing_cols)
print(f"Удалены метрики сопоставления: {existing_cols}")

# Создаём индикаторы для признаков соседей
neighbor_stat_cols = [c for c in X_prepared.columns 
                      if any(x in c for x in ['neighbor_height_mean', 'neighbor_height_median', 
                                               'neighbor_height_min', 'neighbor_height_max',
                                               'neighbor_height_std', 'neighbor_height_q25', 
                                               'neighbor_height_q75'])]

print(f"\nНайдено признаков статистики соседей: {len(neighbor_stat_cols)}")

for col in neighbor_stat_cols:
    # Создаём индикатор: 1 если значение не NaN, иначе 0
    indicator_name = col.replace('neighbor_height', 'has_neighbor_stats')
    X_prepared[indicator_name] = (~X_prepared[col].isna()).astype(int)
    print(f"  {col} → создан индикатор {indicator_name}")

# Заполняем все оставшиеся NaN значением 0
nan_before = X_prepared.isna().sum().sum()
X_prepared = X_prepared.fillna(0)
nan_after = X_prepared.isna().sum().sum()

print(f"\nЗаполнено пропусков: {nan_before - nan_after}")
print(f"Пропусков после обработки: {nan_after}")

print(f"\nИтоговое количество признаков: {X_prepared.shape[1]}")

Удалены метрики сопоставления: ['max_iou', 'mean_iou', 'max_overlap_a', 'max_overlap_b', 'min_dist_m']

Найдено признаков статистики соседей: 14
  neighbor_height_mean_50m → создан индикатор has_neighbor_stats_mean_50m
  neighbor_height_median_50m → создан индикатор has_neighbor_stats_median_50m
  neighbor_height_min_50m → создан индикатор has_neighbor_stats_min_50m
  neighbor_height_max_50m → создан индикатор has_neighbor_stats_max_50m
  neighbor_height_std_50m → создан индикатор has_neighbor_stats_std_50m
  neighbor_height_q25_50m → создан индикатор has_neighbor_stats_q25_50m
  neighbor_height_q75_50m → создан индикатор has_neighbor_stats_q75_50m
  neighbor_height_mean_100m → создан индикатор has_neighbor_stats_mean_100m
  neighbor_height_median_100m → создан индикатор has_neighbor_stats_median_100m
  neighbor_height_min_100m → создан индикатор has_neighbor_stats_min_100m
  neighbor_height_max_100m → создан индикатор has_neighbor_stats_max_100m
  neighbor_height_std_100m → создан инд

#### Шаг 5.5. Запуск кросс-валидации

После обработки пропусков данные готовы для обучения. Для каждого из 6 наборов признаков запускаем 5-fold spatial CV (GroupKFold) и вычисляем метрики:
- **RMSE** — среднеквадратичная ошибка в метрах (штрафует большие ошибки)
- **MAE** — средняя абсолютная ошибка в метрах
- **MSE** — среднеквадратичная ошибка в м²
- **R²** — доля объяснённой дисперсии (чем ближе к 1, тем лучше)

**Важно:** используется пространственная кросс-валидация (GroupKFold) с разбиением по группам `spatial_group` (сетка 1000×1000 м). Это предотвращает пространственное переобучение — здания из одного квартала не попадают одновременно в обучающую и тестовую выборки.

In [11]:
results = {}
# Создаём модель
model = LinearRegression()
for name, features in feature_sets.items():
    # Фильтруем признаки: оставляем только те, что есть в X_prepared
    available_features = [f for f in features if f in X_prepared.columns]
    
    print(f"Набор: {name} (признаков: {len(available_features)})")
    
    X = X_prepared[available_features].copy()
    
    metrics = evaluate_model_cv(X, y, groups, model, cv_folds=5)
    results[name] = metrics
    
    print(f"\nИТОГО для {name}:")
    print(f"  RMSE: {metrics['RMSE_mean']:.3f} ± {metrics['RMSE_std']:.3f} м")
    print(f"  MAE:  {metrics['MAE_mean']:.3f} м")
    print(f"  MSE:  {metrics['MSE_mean']:.3f} м²")
    print(f"  R²:   {metrics['R2_mean']:.3f}")

Набор: 1_Baseline (признаков: 20)
  Fold 1: RMSE=2.510 м, MAE=0.538 м, R²=0.949
  Fold 2: RMSE=2.864 м, MAE=0.611 м, R²=0.927
  Fold 3: RMSE=4.003 м, MAE=0.563 м, R²=0.860
  Fold 4: RMSE=2.348 м, MAE=0.571 м, R²=0.942
  Fold 5: RMSE=2.270 м, MAE=0.515 м, R²=0.949

ИТОГО для 1_Baseline:
  RMSE: 2.799 ± 0.636 м
  MAE:  0.560 м
  MSE:  8.237 м²
  R²:   0.926
Набор: 2_+ n_a_n_b (признаков: 22)
  Fold 1: RMSE=2.503 м, MAE=0.543 м, R²=0.950
  Fold 2: RMSE=2.861 м, MAE=0.622 м, R²=0.927
  Fold 3: RMSE=3.999 м, MAE=0.569 м, R²=0.861
  Fold 4: RMSE=2.348 м, MAE=0.581 м, R²=0.942
  Fold 5: RMSE=2.269 м, MAE=0.526 м, R²=0.949

ИТОГО для 2_+ n_a_n_b:
  RMSE: 2.796 ± 0.635 м
  MAE:  0.568 м
  MSE:  8.222 м²
  R²:   0.926
Набор: 3_+ neighbors_mean (признаков: 22)
  Fold 1: RMSE=2.502 м, MAE=0.638 м, R²=0.950
  Fold 2: RMSE=2.838 м, MAE=0.681 м, R²=0.928
  Fold 3: RMSE=3.987 м, MAE=0.633 м, R²=0.861
  Fold 4: RMSE=2.337 м, MAE=0.653 м, R²=0.943
  Fold 5: RMSE=2.273 м, MAE=0.613 м, R²=0.949

ИТОГО для

## Шаг 6. Итоговая таблица результатов

После завершения кросс-валидации для 6 наборов признаков получены следующие метрики:

- **RMSE** — среднеквадратичная ошибка в метрах (чем меньше, тем лучше)
- **MAE** — средняя абсолютная ошибка в метрах
- **MSE** — среднеквадратичная ошибка в м²
- **R²** — доля объяснённой дисперсии (чем ближе к 1, тем лучше)

**Анализ результатов:**
- Все наборы показывают близкие результаты с RMSE ≈ 2.78–2.80 м
- Базовый набор (20 признаков) уже даёт хорошее качество (R² = 0.924)
- Добавление дополнительных признаков (`n_a/n_b`, соседи) незначительно улучшает или не меняет результат


In [12]:
# Создаём таблицу результатов
results_df = pd.DataFrame(results).T
results_df = results_df[['RMSE_mean', 'RMSE_std', 'MAE_mean', 'MSE_mean', 'R2_mean']]
results_df.columns = ['RMSE (м)', 'RMSE std', 'MAE (м)', 'MSE (м²)', 'R²']

# Форматируем вывод
print("\nРезультаты кросс-валидации (5-fold spatial CV):")
print(results_df.round(3))

# Находим лучший набор по RMSE
best_set = results_df['RMSE (м)'].idxmin()
best_rmse = results_df.loc[best_set, 'RMSE (м)']
best_r2 = results_df.loc[best_set, 'R²']


print(f"Лучший набор по RMSE: {best_set}")
print(f"  RMSE: {best_rmse:.3f} м")




Результаты кросс-валидации (5-fold spatial CV):
                    RMSE (м)  RMSE std  MAE (м)  MSE (м²)     R²
1_Baseline             2.799     0.636    0.560     8.237  0.926
2_+ n_a_n_b            2.796     0.635    0.568     8.222  0.926
3_+ neighbors_mean     2.787     0.631    0.644     8.167  0.926
4_+ neighbors_50m      2.790     0.631    0.642     8.180  0.926
5_+ neighbors_100m     2.788     0.632    0.661     8.171  0.926
6_Full                 2.785     0.632    0.664     8.154  0.926
Лучший набор по RMSE: 6_Full
  RMSE: 2.785 м
